# 04 Joins

Practice joins and identify why data movement can appear even when SQL looks simple.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("module-02-spark-sql").getOrCreate()
base = "../../datasets/module_02"

In [ ]:
municipalities = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/municipalities.csv")
accessibility = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/accessibility_scores.csv")
poi = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/poi_counts.csv")
property_values = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/property_values.csv")
municipalities.printSchema()
municipalities.show(5, truncate=False)

In [ ]:
municipalities.createOrReplaceTempView("municipalities")
accessibility.createOrReplaceTempView("accessibility_scores")
poi.createOrReplaceTempView("poi_counts")
property_values.createOrReplaceTempView("property_values")

In [ ]:
spark.sql("""
SELECT m.municipality_name, m.canton, m.population,
       a.accessibility_score, p.poi_count, v.property_value_index
FROM municipalities m
JOIN accessibility_scores a USING (municipality_id)
JOIN poi_counts p USING (municipality_id)
JOIN property_values v USING (municipality_id)
WHERE m.population >= 25000
ORDER BY v.property_value_index DESC
""").show(20, truncate=False)

In [ ]:
spark.sql("""
EXPLAIN FORMATTED
SELECT m.canton, AVG(v.property_value_index) AS avg_value
FROM municipalities m
JOIN property_values v USING (municipality_id)
GROUP BY m.canton
""").show(truncate=False)